# Learning Rate별 NSMC 정확도 실험

3 epoch 동안 learning rate만 바꾸면서 NSMC 감성 분류 정확도를 비교하는 Colab용 기본 실험 노트북입니다.

- 고정값: seed 42, batch size 32, max length 128, dropout 0.1, AdamW
- 변경값: learning rate `1e-4`, `3e-4`, `5e-4`
- 기록값: epoch별 train/validation loss와 accuracy, best validation checkpoint 기준 test accuracy


## 1. Colab 런타임 준비

Colab에서는 `Runtime > Change runtime type > GPU`를 먼저 선택하세요. 아래 셀은 저장소를 `/content/gpt-lab`에 clone하고 프로젝트 루트로 이동합니다.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_dir = Path("/content/gpt-lab")
    if not repo_dir.exists():
        repo_url = input("GitHub 저장소 URL을 입력하세요 (예: https://github.com/USER/gpt-lab.git): ").strip()
        if repo_url.startswith("github.com/"):
            repo_url = "https://" + repo_url
        if not repo_url:
            raise ValueError("Colab에서는 저장소 URL이 필요합니다.")
        subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
else:
    repo_dir = Path.cwd()

os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("repo_dir:", repo_dir)
print("cwd:", Path.cwd())


In [ ]:
!python --version
!nvidia-smi
!pip install -r requirements.txt


## 2. 환경과 seed 기록

In [ ]:
import json
import platform
import random
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.bpe import BPETokenizer
from src.finetune import GPTForSequenceClassification, ReviewSentimentDataset, evaluate_sentiment, train_epoch_sentiment
from src.model import GPTModel

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("python:", sys.version)
print("python_major_minor:", f"{sys.version_info.major}.{sys.version_info.minor}")
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_version:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("numpy:", np.__version__)
print("device:", device)

try:
    commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    commit = "unknown"
print("git commit:", commit)


## 3. 데이터 준비

In [ ]:
import download_data

paths = download_data.main()

DATA_DIR = Path("data")
LM_TRAIN_PATH = DATA_DIR / "nsmc_lm_train.txt"
TRAIN_JSONL = DATA_DIR / "nsmc_sentiment_train.jsonl"
VAL_JSONL = DATA_DIR / "nsmc_sentiment_val.jsonl"
TEST_JSONL = DATA_DIR / "nsmc_sentiment_test.jsonl"

for path in [LM_TRAIN_PATH, TRAIN_JSONL, VAL_JSONL, TEST_JSONL]:
    print(path, path.exists(), path.stat().st_size if path.exists() else 0)


In [ ]:
def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_data = read_jsonl(TRAIN_JSONL)
val_data = read_jsonl(VAL_JSONL)
test_data = read_jsonl(TEST_JSONL)

def label_summary(rows):
    counts = Counter(row["label"] for row in rows)
    total = len(rows)
    return {"label_0": counts.get(0, 0), "label_1": counts.get(1, 0), "total": total, "positive_ratio": counts.get(1, 0) / total}

print("train:", label_summary(train_data))
print("val:", label_summary(val_data))
print("test:", label_summary(test_data))


## 4. 실험 설정

`USE_SUBSET=True`로 바꾸면 빠른 smoke test를 할 수 있습니다. 최종 기록용 실험은 `USE_SUBSET=False`로 실행하세요.

In [ ]:
LR_CANDIDATES = [1e-4, 3e-4, 5e-4]
NUM_EPOCHS = 3
BATCH_SIZE = 32
MAX_LENGTH = 128
WEIGHT_DECAY = 0.0
VOCAB_SIZE = 3000

USE_SUBSET = False
SUBSET_TRAIN_SIZE = 20000
SUBSET_VAL_SIZE = 4000
SUBSET_TEST_SIZE = 5000

GPT_CONFIG = {
    "vocab_size": VOCAB_SIZE,
    "context_length": MAX_LENGTH,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

print("LR_CANDIDATES:", LR_CANDIDATES)
print("NUM_EPOCHS:", NUM_EPOCHS)
print("BATCH_SIZE:", BATCH_SIZE)
print("MAX_LENGTH:", MAX_LENGTH)
print("USE_SUBSET:", USE_SUBSET)
print("GPT_CONFIG:", GPT_CONFIG)


## 5. BPE tokenizer 학습

In [ ]:
tokenizer_path = DATA_DIR / f"vocab_bpe_{VOCAB_SIZE}.json"
tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE)

if tokenizer_path.exists():
    tokenizer.load(tokenizer_path)
    print("loaded tokenizer:", tokenizer_path)
else:
    corpus = LM_TRAIN_PATH.read_text(encoding="utf-8")
    print("training tokenizer, corpus chars:", len(corpus))
    tokenizer.train(corpus)
    tokenizer.save(tokenizer_path)
    print("saved tokenizer:", tokenizer_path)

print("tokenizer vocab_size:", tokenizer.vocab_size)
print("pad_id:", tokenizer.get_pad_id())


## 6. DataLoader 생성

In [ ]:
if USE_SUBSET:
    train_rows = train_data[:SUBSET_TRAIN_SIZE]
    val_rows = val_data[:SUBSET_VAL_SIZE]
    test_rows = test_data[:SUBSET_TEST_SIZE]
else:
    train_rows = train_data
    val_rows = val_data
    test_rows = test_data

train_dataset = ReviewSentimentDataset(train_rows, tokenizer, max_length=MAX_LENGTH)
val_dataset = ReviewSentimentDataset(val_rows, tokenizer, max_length=MAX_LENGTH)
test_dataset = ReviewSentimentDataset(test_rows, tokenizer, max_length=MAX_LENGTH)

def make_loaders(seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders()

print("train examples:", len(train_dataset))
print("val examples:", len(val_dataset))
print("test examples:", len(test_dataset))
print("train batches:", len(train_loader))


## 7. Learning rate별 3 epoch 실험

In [ ]:
def build_model():
    gpt = GPTModel(GPT_CONFIG)
    model = GPTForSequenceClassification(
        gpt_model=gpt,
        num_labels=2,
        drop_rate=GPT_CONFIG["drop_rate"],
        pad_id=tokenizer.get_pad_id(),
    )
    return model.to(device)

def run_lr_experiment(lr):
    set_seed(SEED)
    train_loader, val_loader, test_loader = make_loaders(seed=SEED)
    model = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_epoch = 0
    best_path = Path("checkpoints") / f"sentiment_lr_{lr:.0e}_best.pt"
    best_path.parent.mkdir(exist_ok=True)
    history = []

    start_time = time.time()
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_epoch_sentiment(model, train_loader, optimizer, device)
        val_loss, val_acc = evaluate_sentiment(model, val_loader, device)
        elapsed = time.time() - start_time

        row = {
            "lr": lr,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "elapsed_sec": elapsed,
        }
        history.append(row)

        print(
            f"lr={lr:.0e} epoch={epoch}/{NUM_EPOCHS} "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
            f"elapsed={elapsed/60:.1f}m"
        )

        if val_acc > best_val_acc or (val_acc == best_val_acc and val_loss < best_val_loss):
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch
            torch.save(model.state_dict(), best_path)

    model.load_state_dict(torch.load(best_path, map_location=device))
    test_loss, test_acc = evaluate_sentiment(model, test_loader, device)
    total_elapsed = time.time() - start_time

    summary = {
        "lr": lr,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "elapsed_sec": total_elapsed,
        "checkpoint": str(best_path),
    }
    print("BEST/TEST:", summary)
    return history, summary

all_history = []
summaries = []

for lr in LR_CANDIDATES:
    print("\n" + "=" * 80)
    print(f"Running learning_rate={lr:.0e}")
    history, summary = run_lr_experiment(lr)
    all_history.extend(history)
    summaries.append(summary)

print("\n완료")


## 8. 결과 표와 그래프

In [ ]:
def print_markdown_table(rows, columns):
    print("| " + " | ".join(columns) + " |")
    print("| " + " | ".join(["---"] * len(columns)) + " |")
    for row in rows:
        values = []
        for col in columns:
            value = row[col]
            if isinstance(value, float):
                if "acc" in col or "loss" in col:
                    value = f"{value:.4f}"
                elif col == "lr":
                    value = f"{value:.0e}"
                elif col == "elapsed_sec":
                    value = f"{value/60:.1f}m"
            values.append(str(value))
        print("| " + " | ".join(values) + " |")

summary_columns = ["lr", "best_epoch", "best_val_loss", "best_val_acc", "test_loss", "test_acc", "elapsed_sec", "checkpoint"]
print_markdown_table(summaries, summary_columns)

best = max(summaries, key=lambda row: (row["best_val_acc"], -row["best_val_loss"]))
print("\nBest learning_rate:", f"{best['lr']:.0e}")
print("Best validation accuracy:", f"{best['best_val_acc']:.4f}")
print("Test accuracy:", f"{best['test_acc']:.4f}")


In [ ]:
plt.figure(figsize=(8, 4))
for lr in LR_CANDIDATES:
    rows = [row for row in all_history if row["lr"] == lr]
    epochs = [row["epoch"] for row in rows]
    val_accs = [row["val_acc"] for row in rows]
    plt.plot(epochs, val_accs, marker="o", label=f"lr={lr:.0e}")

plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Learning rate comparison")
plt.xticks(range(1, NUM_EPOCHS + 1))
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
for lr in LR_CANDIDATES:
    rows = [row for row in all_history if row["lr"] == lr]
    epochs = [row["epoch"] for row in rows]
    val_losses = [row["val_loss"] for row in rows]
    plt.plot(epochs, val_losses, marker="o", label=f"lr={lr:.0e}")

plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Learning rate comparison")
plt.xticks(range(1, NUM_EPOCHS + 1))
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 9. 결과 저장

실험 결과는 `results/lr_accuracy_experiment.json`에 저장됩니다. 최종 보고서에는 위 markdown 표를 복사해 넣으면 됩니다.

In [ ]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
result_path = results_dir / "lr_accuracy_experiment.json"

result = {
    "seed": SEED,
    "git_commit": commit,
    "device": str(device),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "weight_decay": WEIGHT_DECAY,
    "gpt_config": GPT_CONFIG,
    "use_subset": USE_SUBSET,
    "history": all_history,
    "summaries": summaries,
    "best": best,
}

with result_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("saved:", result_path)


# 1. 작은 smoke test를 1회 돌린다.